# 🔬 Lesson 16 — Molecular Dynamics Analysis

> **Author:** Omar Hamouda  
> **Environment:** Python 3.11 | WSL/Ubuntu | Conda (biochem)  
> **Purpose:** Analyze the MD simulation of Trypsin-13U complex

---

## 📌 Overview

After docking (Lesson 14) and ADMET screening (Lesson 15), we ran a **Molecular Dynamics (MD) simulation** on the Trypsin-13U complex using **OpenMM** with **AMBER** force fields.

This notebook analyzes the simulation results:

1. **RMSD** — Did the protein stay stable?
2. **RMSF** — Which residues are flexible?
3. **MMPBSA** — What is the binding free energy?
4. **Hydrogen Bonds** — Which H-bonds are stable?
5. **ProLIF** — Which interactions persist during the simulation?

---

## 🧬 Scientific Context

**Docking** gives a score for a particular pose under the chosen scoring function; it is not an experimental affinity.

**MD Simulation** answers the dynamic question — *does the ligand stay bound over time?*

| Method | Question | Output |
|--------|----------|--------|
| **Docking** | How favorable is the scored pose? | Docking score (kcal/mol) |
| **MD** | Does it stay? | RMSD, RMSF, trajectories |
| **MM/GBSA** | What is the model-based interaction estimate? | Estimated ΔG (kcal/mol) |

---

## 📁 Required Files

All files are in `md/colab_workshop/`:

| File | Location | Description |
|------|----------|-------------|
| `production_log.csv` | `01_md_simulation/4. production/` | Energy, temperature over time |
| `FINAL_RESULTS.dat` | `02_analysis/1. mmpbsa/` | MMPBSA binding energy |
| `per_residue_FINAL.csv` | `02_analysis/1. mmpbsa/` | Per-residue energy decomposition |
| `hbonds_correct.csv` | `02_analysis/2. hbonds/` | Hydrogen bond analysis |
| `prolif_summary.csv` | `02_analysis/3. prolif/` | Interaction fingerprints |
| `rmsd_plot.png` | `03_figures/` | Pre-generated RMSD plot |
| `rmsf_plot.png` | `03_figures/` | Pre-generated RMSF plot |

In [1]:
# Auto-Fix Working Directory 
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working directory set to:", os.getcwd())

Working directory set to: /home/omar/biochemist-python_ORGANIZED


---

## 1️⃣ Setup & Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.2)

# Base paths
MD_BASE = os.path.join("md", "colab_workshop")
SIM_DIR = os.path.join(MD_BASE, "01_md_simulation")
ANA_DIR = os.path.join(MD_BASE, "02_analysis")
FIG_DIR = os.path.join(MD_BASE, "03_figures")

print("✅ All imports successful.")
print(f"   MD base: {os.path.abspath(MD_BASE)}")

---

## 2️⃣ MD Simulation Summary

The MD simulation was performed using **OpenMM** with the following protocol:

| Stage | Description | Details |
|-------|-------------|---------|
| **1. System Prep** | Build the solvated complex | AMBER ff14SB + GAFF2, TIP3P water |
| **2. Minimization** | Remove bad contacts | Energy minimization |
| **3. Equilibration** | Heat the system | NVT → NPT at 300K |
| **4. Production** | Collect data | NPT, 300K, ~500 frames |

**Complex:** Trypsin (2ZQ2) + Ligand 13U  
**Force Field:** AMBER ff14SB (protein) + GAFF2 (ligand)  
**Water Model:** TIP3P  

---

## 3️⃣ Production Run — Energy & Temperature

Let's verify the simulation was stable by checking the **energy** and **temperature** over time.

In [ ]:
# Load production log
prod_log = pd.read_csv(
    os.path.join(SIM_DIR, "4. production", "production_log.csv"),
    
)

# Clean column names (remove quotes and extra spaces)
prod_log.columns = [c.strip().strip('"#') for c in prod_log.columns]

print(f"Simulation frames: {len(prod_log)}")
print(f"Columns: {list(prod_log.columns)}")
prod_log.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Total Energy ---
time_col = [c for c in prod_log.columns if 'Time' in c][0]
time_ns = prod_log[time_col] / 1000  # ps → ns

energy_col = [c for c in prod_log.columns if 'Total Energy' in c][0]
axes[0].plot(time_ns, prod_log[energy_col] / 1000, color='steelblue', alpha=0.7)
axes[0].set_xlabel('Time (ns)')
axes[0].set_ylabel('Total Energy (× 10³ kJ/mol)')
axes[0].set_title('Total Energy')

# --- Temperature ---
temp_col = [c for c in prod_log.columns if 'Temperature' in c][0]
axes[1].plot(time_ns, prod_log[temp_col], color='crimson', alpha=0.7)
axes[1].axhline(y=300, color='black', linestyle='--', label='Target: 300K')
axes[1].set_xlabel('Time (ns)')
axes[1].set_ylabel('Temperature (K)')
axes[1].set_title('Temperature')
axes[1].legend()

# --- Density ---
density_col = [c for c in prod_log.columns if 'Density' in c][0]
axes[2].plot(time_ns, prod_log[density_col], color='forestgreen', alpha=0.7)
axes[2].set_xlabel('Time (ns)')
axes[2].set_ylabel('Density (g/mL)')
axes[2].set_title('Density')

plt.suptitle('MD Production Run — Simulation Stability', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join('figures', 'md_stability.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: figures/md_stability.png")

### 🔍 Interpretation

- **Temperature**: should fluctuate around the target (~300 K) rather than drift systematically.
- **Density**: should be reasonably stable for the chosen water/thermostat/barostat setup; it is a diagnostic, not proof of equilibration.
- **Energy**: fluctuations are expected in an NPT simulation; a perfectly flat energy trace is not required. Large systematic drift would be more concerning.

---

## 4️⃣ RMSD — Structural Stability

**RMSD** (Root Mean Square Deviation) measures structural deviation from a reference structure after the analysis alignment.

- A **plateau** can indicate that the system has reached a relatively stable dynamical regime.
- An initial rise can reflect relaxation from the starting structure and does **not** by itself mean unfolding.
- Large sustained changes can motivate further structural inspection, but RMSD alone cannot diagnose unfolding or prove ligand retention.

In [ ]:
# Display the pre-generated RMSD plot
rmsd_path = os.path.join(FIG_DIR, "rmsd_plot.png")

fig, ax = plt.subplots(figsize=(10, 6))
img = mpimg.imread(rmsd_path)
ax.imshow(img)
ax.axis('off')
ax.set_title('RMSD Over Time — Trypsin-13U Complex', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("💡 Look for: RMSD plateau after initial equilibration.")
print("   A plateau can indicate a relatively stable dynamical regime in the analyzed window; RMSD alone does not establish equilibrium.")

---

## 5️⃣ RMSF — Residue Flexibility

**RMSF** (Root Mean Square Fluctuation) describes the average positional fluctuation of each residue around its mean position.

- Loops and termini often show higher RMSF.
- Lower RMSF at a binding-site residue indicates lower positional fluctuation in this trajectory, not necessarily a stronger ligand interaction.
- RMSF should therefore be interpreted together with contacts, hydrogen bonds, and other structural metrics.

In [ ]:
# Display the pre-generated RMSF plot
rmsf_path = os.path.join(FIG_DIR, "rmsf_plot.png")

fig, ax = plt.subplots(figsize=(12, 5))
img = mpimg.imread(rmsf_path)
ax.imshow(img)
ax.axis('off')
ax.set_title('RMSF Per Residue — Trypsin', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("💡 Key binding site residues (ASP171, SER192, SER172, GLY196)")
print("   should be interpreted as local mobility measures; low RMSF alone does not establish stronger ligand binding.")

---

## 6️⃣ MM/GBSA — Model-Based Binding-Energy Estimate

The stored analysis uses **MMPBSA.py with the Generalized Born model (`igb=5`)**. It is therefore more precisely described as an **MM/GBSA-style estimate**, not a Poisson–Boltzmann calculation.

$$\Delta G_{est} = G_{complex} - G_{receptor} - G_{ligand}$$

Stored run summary:

- **ΔG estimate:** `-32.13 ± 17.22 kcal/mol`
- **Sampled complex frames:** `50.9`
- **Salt concentration:** `0.150 M`
- **GB model:** `igb=5`

A negative value is favorable **within this computational model**, but it should not be interpreted as an experimental binding free energy or used alone to rank compounds quantitatively. The standard deviation is also substantial.

In [ ]:
# Read MMPBSA final results
mmpbsa_path = os.path.join(ANA_DIR, "1. mmpbsa", "FINAL_RESULTS.dat")

with open(mmpbsa_path, 'r') as f:
    content = f.read()

# Parse the DELTA section for binding energy
lines = content.split('\n')
delta_section = False
results = {}

for line in lines:
    if 'Differences (Complex - Receptor - Ligand):' in line:
        delta_section = True
        continue
    if delta_section and 'DELTA TOTAL' in line:
        parts = line.split()
        results['DELTA TOTAL'] = float(parts[2])
        results['Std Dev'] = float(parts[3])
    elif delta_section and 'VDWAALS' in line:
        parts = line.split()
        results['VDW'] = float(parts[1])
    elif delta_section and 'EEL' in line and 'EELEC' not in content[:100]:
        parts = line.split()
        results['Electrostatic'] = float(parts[1])

print("═" * 60)
print("  MMPBSA Results — Trypsin + 13U")
print("═" * 60)

if 'DELTA TOTAL' in results:
    dg = results['DELTA TOTAL']
    sd = results['Std Dev']
    print(f"\n  ΔG binding = {dg:.2f} ± {sd:.2f} kcal/mol")
    print(f"\n  Interpretation:")
    if dg < -10:
        print("  ✅ Strong binding — ligand is tightly bound!")
    elif dg < -5:
        print("  ✅ Moderate binding — reasonable complex.")
    elif dg < 0:
        print("  ⚠️  Weak binding — ligand may dissociate.")
    else:
        print("  ❌ Unfavorable — ligand will leave.")
else:
    print("\n  Could not parse DELTA TOTAL. Showing raw file:")
    # Show the relevant section
    in_delta = False
    for line in lines:
        if 'Differences' in line:
            in_delta = True
        if in_delta:
            print(" ", line)

print("\n" + "═" * 60)

---

## 7️⃣ Per-Residue Energy Decomposition

Which residues contribute most to binding?

This tells us **which amino acids are most important** for holding the ligand.

In [ ]:
# Load per-residue decomposition
decomp_path = os.path.join(ANA_DIR, "1. mmpbsa", "per_residue_FINAL.csv")
decomp = pd.read_csv(decomp_path, encoding='utf-8')

# Clean column names
decomp.columns = [c.strip() for c in decomp.columns]
# Identify the energy column (might have special chars)
energy_col = [c for c in decomp.columns if 'G' in c or 'kcal' in c][0]
res_col = decomp.columns[0]

# Exclude the ligand itself and filter significant contributions
decomp_prot = decomp[~decomp[res_col].str.contains('UNK', na=False)].copy()
decomp_prot = decomp_prot.sort_values(energy_col)

# Top 15 contributing residues
top15 = decomp_prot.head(15)

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#d32f2f' if v < -1 else '#1976d2' if v < -0.5 else '#90a4ae' 
          for v in top15[energy_col]]
bars = ax.barh(range(len(top15)), top15[energy_col], color=colors)
ax.set_yticks(range(len(top15)))
ax.set_yticklabels(top15[res_col])
ax.set_xlabel('ΔG contribution (kcal/mol)')
ax.set_title('Top 15 Residues Contributing to Binding — MMPBSA Decomposition', fontweight='bold')
ax.axvline(x=0, color='black', linewidth=0.5)
ax.invert_yaxis()

# Add value labels
for i, (v, bar) in enumerate(zip(top15[energy_col], bars)):
    ax.text(v - 0.05, i, f'{v:.2f}', va='center', ha='right', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join('figures', 'mmpbsa_decomposition.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: figures/mmpbsa_decomposition.png")

### 🔍 Per-residue decomposition

The decomposition highlights residues with large **computed energetic contributions in this model**. It is not a direct experimental measure of residue binding strength or proof of causal importance.

| Residue | ΔG contribution (kcal/mol) |
|---------|-----------------------------:|
| **TRP193** | -3.03 |
| **LEU81** | -1.54 |
| **CYX173** | -1.00 |
| **CYX197** | -0.96 |
| **HID40** | -0.91 |
| **GLN174** | -0.86 |

---

## 8️⃣ Hydrogen Bond Analysis

Hydrogen bonds can contribute to ligand stabilization within a protein binding site. Higher occupancy indicates that an observed hydrogen bond persists for a larger fraction of the simulation.

In [ ]:
# Load hydrogen bond data
hbonds_path = os.path.join(ANA_DIR, "2. hbonds", "hbonds_correct.csv")
hbonds = pd.read_csv(hbonds_path)

print("═" * 60)
print("  Hydrogen Bonds — 13U with Trypsin")
print("═" * 60)
print()
print(hbonds.to_string(index=False))
print()

In [ ]:
# Visualize H-bond occupancies
occ_col = [c for c in hbonds.columns if 'Occupancy' in c or 'ccupancy' in c][0]
donor_col = hbonds.columns[0]
acceptor_col = hbonds.columns[1]

hbonds['Label'] = hbonds[donor_col] + ' → ' + hbonds[acceptor_col]
hbonds_sorted = hbonds.sort_values(occ_col, ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#2e7d32' if v >= 80 else '#f57f17' if v >= 50 else '#c62828' 
          for v in hbonds_sorted[occ_col]]
ax.barh(hbonds_sorted['Label'], hbonds_sorted[occ_col], color=colors)
ax.set_xlabel('Occupancy (%)')
ax.set_title('Hydrogen Bond Occupancy — 13U with Trypsin', fontweight='bold')
ax.axvline(x=80, color='green', linestyle='--', alpha=0.5, label='Strong (>80%)')
ax.axvline(x=50, color='orange', linestyle='--', alpha=0.5, label='Moderate (>50%)')
ax.legend()

plt.tight_layout()
plt.savefig(os.path.join('figures', 'hbonds_occupancy.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: figures/hbonds_occupancy.png")

### 🔍 Interpretation

The hydrogen-bond analysis reports:

- **ASP171** — 100.0% occupancy
- **SER192** — 99.8% occupancy
- **SER172** — 99.4% occupancy
- **GLY196** — 89.2% occupancy for the listed ligand→protein donor pair
- **GLY196** — 87.2% occupancy for the reciprocal protein→ligand pair

High occupancy means that the geometric/chemical criterion used by the analysis was satisfied frequently in this trajectory. It supports persistent contacts **in this simulation**, but it does not by itself establish binding affinity or experimental importance.

---

## 9️⃣ ProLIF Interaction Fingerprint

**ProLIF** (Protein-Ligand Interaction Fingerprints) identifies interaction types between the ligand and protein residues across the MD trajectory and reports their occupancy in the stored analysis.

In [ ]:
# Load ProLIF summary
prolif_path = os.path.join(ANA_DIR, "3. prolif", "prolif_summary.csv")
prolif = pd.read_csv(prolif_path)

# Parse interaction details
int_col = prolif.columns[0]
occ_col = prolif.columns[1]

# Extract residue name and interaction type
prolif['Residue'] = prolif[int_col].apply(
    lambda x: x.split("'")[3] if "'" in str(x) else x
)
prolif['Interaction'] = prolif[int_col].apply(
    lambda x: x.split("'")[5] if "'" in str(x) else x
)

# Show interactions observed at ≥30% occupancy in this stored trajectory
significant = prolif[prolif[occ_col] >= 30].sort_values(occ_col, ascending=False)

print("═" * 60)
print("  ProLIF — Interactions observed at ≥30% occupancy")
print("═" * 60)
for _, row in significant.iterrows():
    bar = '█' * int(row[occ_col] / 5)
    print(f"  {row['Residue']:12s} {row['Interaction']:15s} {row[occ_col]:5.1f}% {bar}")
print()

In [ ]:
# Pivot: Residue vs Interaction Type heatmap
pivot = prolif.pivot_table(
    index='Residue', columns='Interaction', values=occ_col, aggfunc='max'
).fillna(0)

# Keep only residues with at least one significant interaction
pivot = pivot[pivot.max(axis=1) >= 20]
pivot = pivot.sort_values(pivot.columns[0], ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    pivot, annot=True, fmt='.0f', cmap='YlOrRd',
    linewidths=0.5, ax=ax, vmin=0, vmax=100,
    cbar_kws={'label': 'Occupancy (%)'}
)
ax.set_title('Protein-Ligand Interaction Fingerprint — MD Trajectory', fontweight='bold')
ax.set_ylabel('Residue')
ax.set_xlabel('Interaction Type')

plt.tight_layout()
plt.savefig(os.path.join('figures', 'prolif_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: figures/prolif_heatmap.png")

---

## 🔟 Final Summary

Let's bring all the analyses together.

In [ ]:
# Final summary table
print("\n" + "═" * 70)
print("  FINAL SUMMARY — Trypsin (2ZQ2) + Ligand 13U")
print("═" * 70)

print("\n  📊 Simulation:")
print(f"     Frames analyzed: {len(prod_log)}")
sim_time = prod_log[time_col].max() / 1000
print(f"     Simulation time: {sim_time:.1f} ns")

print(f"\n  🔬 MM/GBSA-style Energy Estimate:")
if 'DELTA TOTAL' in results:
    print(f"     ΔG = {results['DELTA TOTAL']:.2f} ± {results['Std Dev']:.2f} kcal/mol")

print(f"\n  🤝 Hydrogen Bonds:")
print(f"     Total H-bonds found: {len(hbonds)}")
strong = hbonds[hbonds[hbonds.columns[4]] >= 80] if len(hbonds.columns) > 4 else hbonds
print(f"     Strong (>80% occupancy): {len(strong)}")

print(f"\n  🎯 Residues highlighted by energetic decomposition and/or interaction analyses:")
key_residues = ['ASP171', 'SER192', 'SER172', 'GLY196', 'TRP193', 'HID40']
for res in key_residues:
    print(f"     • {res}")

print(f"\n  ✅ Conclusion:")
print(f"     13U shows sustained interaction features in this historical simulation.")
print(f"     The MD results provide computational support for")
print(f"     recurrent contacts in this historical simulation, but do not")
print(f"     experimentally validate the current docking ranking; the MM/GBSA estimate is model-dependent.")

---

## 📋 What We Learned

| Analysis | Result | Meaning |
|----------|--------|---------|
| **RMSD** | Stable-looking trajectory region | Consistent with limited global structural drift in the analyzed trajectory |
| **RMSF** | Local fluctuation pattern | Describes residue mobility; not direct evidence of binding strength |
| **MM/GBSA** | -32.13 ± 17.22 kcal/mol | Favorable model-based estimate with substantial uncertainty |
| **H-bonds** | ASP171, SER192 ~100% | Persistent geometric contacts in this simulation |
| **ProLIF** | Several interactions ≥30% occupancy | Recurrent interaction patterns in this trajectory |

---

## 🎯 Full Pipeline Summary

```
Lessons 01–08: Python + Data Analysis (tools)
     ↓
Lessons 09–11: Protein Structures (target)
     ↓
Lesson 12: Ligand Discovery → 117 ligands found
     ↓
Lesson 13: Binding Site → Trypsin 2ZQ2 analyzed
     ↓
Lesson 14: Standardized docking → Current Top 5: R11, 13U, BAH, 12U, 607
     ↓
Lesson 15: ADMET → Rule-based drug-likeness assessment for the current Top 5
     ↓
Lesson 16: Historical MD analysis → 13U shows stable behavior in the simulated binding site

**🏆 Conclusion:** Starting from the Trypsin structure (2ZQ2), we screened 117 candidate ligands using standardized docking, evaluated the current Top 5 with rule-based drug-likeness descriptors, and analyzed a historical Trypsin–13U MD simulation. The MD results support stable behavior of 13U in that simulated complex but do not experimentally validate the current docking ranking.
---

*End of Workshop — Congratulations!* 🎓